# Exercise 21.2: Scaled Dot-Product Attention

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuhung16au/Maths-for-IT-Notebooks/blob/main/Chapter-21/02-scaled-dot-product-attention.ipynb)

- **Book Reference:** Chapter 21 (*Mathematics of Transformers and Large Language Models*, Section 21.2)
- **Estimated Time:** 15–20 minutes
- **Learning Objectives:**
  1. Implement numerically stable row-wise `softmax`.
  2. Implement scaled dot-product attention $\operatorname{Attention}(Q, K, V) = \operatorname{softmax}\!\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$.
  3. Verify the Chapter 21 2-token worked example ($0.731$ and $0.269$ weights, output $3.076$).

In [ ]:
import numpy as np

## Mathematical Background

Given Query $Q \in \mathbb{R}^{T \times d_k}$, Key $K \in \mathbb{R}^{T \times d_k}$, and Value $V \in \mathbb{R}^{T \times d_v}$:
$$\operatorname{Attention}(Q, K, V) = \operatorname{softmax}\!\left(\frac{Q K^T}{\sqrt{d_k}}\right) V$$

### Worked Example Check (Chapter 21.2)
With $d_k = 1$, $Q = \begin{bmatrix} 1 \\ 0 \end{bmatrix}$, $K = \begin{bmatrix} 1 \\ 0 \end{bmatrix}$, $V = \begin{bmatrix} 2 \\ 6 \end{bmatrix}$:
- $Q K^T = \begin{bmatrix} 1 & 0 \\ 0 & 0 \end{bmatrix}$
- Row 0 softmax: $\left[\frac{e}{e+1}, \frac{1}{e+1}\right] \approx [0.73106, 0.26894]$
- Output for token 1: $0.73106(2) + 0.26894(6) \approx 3.07577$.

In [ ]:
def stable_softmax(x: np.ndarray, axis: int = -1) -> np.ndarray:
    """Compute numerically stable softmax along the given axis.

    Args:
        x: Input array of logits.
        axis: Axis along which to normalise.

    Returns:
        Probability array of the same shape as x summing to 1 along `axis`.
    """
    # TODO: Subtract np.max(x, axis=axis, keepdims=True) before exp and normalise
    raise NotImplementedError("Complete this TODO")


def scaled_dot_product_attention(
    Q: np.ndarray,
    K: np.ndarray,
    V: np.ndarray,
) -> tuple[np.ndarray, np.ndarray]:
    """Compute single-head scaled dot-product attention.

    Args:
        Q: Query matrix of shape (..., T_q, d_k).
        K: Key matrix of shape (..., T_k, d_k).
        V: Value matrix of shape (..., T_k, d_v).

    Returns:
        Tuple (output, weights) where output has shape (..., T_q, d_v)
        and weights has shape (..., T_q, T_k).
    """
    # TODO: Compute scores = (Q @ swapaxes(K, -1, -2)) / sqrt(d_k), weights = stable_softmax(scores), output = weights @ V
    raise NotImplementedError("Complete this TODO")

<details>
<summary>Hint</summary>

- Use `np.swapaxes(K, -1, -2)` instead of `K.T` so the function works for both 2D `(T, d_k)` and batched 3D `(H, T, d_k)` tensors.
- `d_k = K.shape[-1]` gives the key dimension regardless of leading batch/head dimensions.

</details>

In [ ]:
# 1. Chapter 21 worked example (2 tokens, d_k=1)
Q_ex = np.array([[1.0], [0.0]])
K_ex = np.array([[1.0], [0.0]])
V_ex = np.array([[2.0], [6.0]])

out_ex, w_ex = scaled_dot_product_attention(Q_ex, K_ex, V_ex)
expected_w0 = np.array([np.e / (np.e + 1.0), 1.0 / (np.e + 1.0)])
assert np.allclose(w_ex[0], expected_w0, atol=1e-6)
assert np.allclose(w_ex[1], [0.5, 0.5])
assert np.isclose(out_ex[0, 0], expected_w0[0] * 2.0 + expected_w0[1] * 6.0)
assert np.allclose(w_ex.sum(axis=-1), 1.0)

# 2. Exercise 21.3 check: weights [0.8, 0.2] and values [10, 20] -> 12.0
assert np.isclose(np.array([0.8, 0.2]) @ np.array([10.0, 20.0]), 12.0)

print(f"All checks passed! Token 1 weights={np.round(w_ex[0], 4)}, output={out_ex[0, 0]:.4f}")

## Think Deeper

1. Why do we divide $Q K^T$ by $\sqrt{d_k}$ before applying `softmax` (Exercise 21.2)?
2. What happens to the variance of $q \cdot k$ when $q, k \sim \mathcal{N}(0, I_{d_k})$ if we omit the $\sqrt{d_k}$ scaling as $d_k$ grows large (e.g. $d_k = 512$)?